# SOXX 发行人折溢价原始页面覆盖预检

只核对发行人两个实际HTML响应的逐日折溢价数组及NAS身份；不从 NAV 与其他价格源拼出未认证 premium，不读取收益或拟合。

In [1]:
from pathlib import Path
import json,hashlib,html,re
root=Path.cwd();card=root/'research/soxx_issuer_premium_source_probe_20261022'
plan=json.loads((card/'plan.json').read_text());result=json.loads((card/'source_probe_result.json').read_text())
assert result['plan_sha256']==hashlib.sha256((card/'plan.json').read_bytes()).hexdigest()
observed=[]
for entry in result['sources']:
    path=root/entry['path'];raw=path.read_bytes()
    assert hashlib.sha256(raw).hexdigest()==entry['sha256'] and len(raw)==entry['bytes']
    text=html.unescape(raw.decode('utf-8',errors='replace'))
    offset=text.find('premiumDiscountChartData');assert offset>=0
    part=text[offset:offset+25000]
    dates=[int(x) for x in re.search(r'"asOfDate":\[(.*?)\]',part).group(1).split(',')]
    values=json.loads('['+re.search(r'"value":\[(.*?)\]',part).group(1)+']')
    assert len(dates)==len(values)==entry['embedded_daily_rows']
    assert str(dates[0])==entry['first_date'] and str(dates[-1])==entry['last_date']
    observed.append((dates,values))
assert observed[0]==observed[1] and len(observed[0][0])==452
archive=json.loads((card/'nas_source_archive_receipt.json').read_text())
assert archive['remote_hash_verified'] and archive['archive_sha256']=='3d31b8780744556780b84023f9418331da345019feb6872962f4f336e8ecfe50'
print({'status':result['status'],'first':observed[0][0][0],'last':observed[0][0][-1],'rows':len(observed[0][0]),'model_fits':0,'accounts':0})

{'status': 'current_issuer_premium_html_2025_2026_only_historical_endpoint_missing', 'first': 20250102, 'last': 20261021, 'rows': 452, 'model_fits': 0, 'accounts': 0}


## 中文结论

两个发行人原始HTML页面均只含2025-01-02至2026-10-21的452个逐日折溢价值；`asOfDate=20161230` 参数没有改变页面数组。原字节与NAS哈希已核对，但没有得到2011–2022完整训练源或原始公布时钟。决定：`needs_original_full_history_and_publication_clock_no_factor`，0拟合、0账户。